## Library

In [ ]:
import polars as pl
from datetime import datetime
import pickle

In [ ]:
!gdown 1VcFzVplwLSMHPM8i3A5qff7nTOp77R5c
# !rm transaction_full_2025.parquet

In [ ]:
!gdown 1M-8S_xsMBVeKqEPQnNYQCni2kb6ND-fQ
# !rm event_full_2025.parquet

In [ ]:
# !gdown 1S8abCHVS9COuWhPDwBI-xtI1SnnPmBgA
# !rm items.parquet

# Pre-pros before feature-eng

In [ ]:
pl.Config.set_streaming_chunk_size(1_000_000)

In [ ]:
transaction = pl.scan_parquet("transaction_full_2025.parquet")
event = pl.scan_parquet("event_full_2025.parquet")

In [ ]:
transaction.head().collect(streaming=True)

In [ ]:
event.head().collect(streaming=True)

In [ ]:
# Chuẩn hóa bảng transaction
tx = (
    transaction
    .select([
        "customer_id",
        "item_id",
        "price",
        "quantity",
        "event_type",
        "updated_date",
        "discount",          # giữ discount gốc từ transaction
    ])
)

# Chuẩn hóa bảng event
ev = (
    event
    .select([
        "customer_id",
        "item_id",
        "price",
        "quantity",
        "event_type",
        "updated_date",
        pl.lit(0).alias("discount"),   # event không có discount => gán 0
    ])
)

# df = pl.concat(
#         [tx, ev],
#         how="vertical_relaxed"
#       )
df = tx

In [ ]:
print(df.head().collect(engine="streaming"))

# Normalizing df

In [ ]:
df = (
    df
    .with_columns([
        pl.col("event_type").str.to_lowercase().alias("event_type"),
        pl.col("price").cast(pl.Float32).alias("price"),
        pl.col("quantity").fill_null(1).cast(pl.Int32).alias("quantity"),
        pl.col("discount").fill_null(0).cast(pl.Float32).alias("discount"),
    ])
    .with_columns([
        pl.when(pl.col("event_type") == "view_item").then(1)
        .when(pl.col("event_type") == "add_to_cart").then(2)
        .when(pl.col("event_type") == "purchase").then(5)
        .otherwise(1)
        .alias("event_weight")
    ])
)

In [ ]:
print(df.head().collect(engine="streaming"))

In [ ]:
n_customers = (
    df
    .select(pl.col("customer_id").n_unique())
    .collect(engine="streaming")
    .item()
)

print(n_customers)

## Data split

In [ ]:
# Nếu df đang là LazyFrame thì dùng luôn

# Train model:
# history_train: dùng dữ liệu trước tháng 12
# label_dec: dùng purchase trong tháng 12 làm groundtruth
history_train = df.filter(
    pl.col("updated_date") < datetime(2025, 12, 1)
)

label_dec = df.filter(
    (pl.col("updated_date") >= datetime(2025, 12, 1)) &
    (pl.col("updated_date") < datetime(2026, 1, 1))
)

# Predict tháng 1:
# dùng toàn bộ lịch sử trước tháng 1
history_jan = df.filter(
    pl.col("updated_date") < datetime(2026, 1, 1)
)

## Feature Eng

In [ ]:
rec_time_train = datetime(2025, 12, 1)

### Users Feature

In [ ]:
# rec_time = datetime(2025, 11, 1)

user_features = (
    history_train
    .group_by("customer_id")
    .agg([
        pl.len().alias("user_total_events"),

        (pl.col("event_type") == "view_item").sum().alias("user_view_count"),
        (pl.col("event_type") == "add_to_cart").sum().alias("user_cart_count"),
        (pl.col("event_type") == "purchase").sum().alias("user_purchase_count"),

        pl.col("item_id").n_unique().alias("user_unique_items"),

        pl.col("price").mean().alias("user_avg_price"),
        pl.col("price").max().alias("user_max_price"),
        pl.col("price").min().alias("user_min_price"),

        pl.col("quantity").sum().alias("user_total_quantity"),
        pl.col("event_weight").sum().alias("user_total_weight"),

        (pl.col("price") * pl.col("quantity")).sum().alias("user_total_spending"),

        pl.col("discount").mean().alias("user_avg_discount"),

        pl.col("updated_date").max().alias("user_last_time"),
    ])
    .with_columns([
        (
            (pl.lit(rec_time_train) - pl.col("user_last_time"))
            .dt.total_seconds() / 86400
        ).alias("user_recency_days"),

        (
            pl.col("user_purchase_count") /
            (pl.col("user_total_events") + 1)
        ).alias("user_purchase_ratio"),

        (
            pl.col("user_cart_count") /
            (pl.col("user_view_count") + 1)
        ).alias("user_cart_per_view"),
    ])
    .drop("user_last_time")
)

user_features.sink_parquet(
    "/kaggle/working/user_features_train.parquet",
    compression="zstd"
)

In [ ]:
print(user_features.head().collect(engine="streaming"))

### Item feature

In [ ]:
item_features = (
    history_train
    .group_by("item_id")
    .agg([
        pl.len().alias("item_total_events"),

        (pl.col("event_type") == "view_item").sum().alias("item_view_count"),
        (pl.col("event_type") == "add_to_cart").sum().alias("item_cart_count"),
        (pl.col("event_type") == "purchase").sum().alias("item_purchase_count"),

        pl.col("customer_id").n_unique().alias("item_unique_users"),

        pl.col("price").mean().alias("item_avg_price"),
        pl.col("price").max().alias("item_max_price"),
        pl.col("price").min().alias("item_min_price"),

        pl.col("quantity").sum().alias("item_total_quantity"),
        pl.col("event_weight").sum().alias("item_total_weight"),

        (pl.col("price") * pl.col("quantity")).sum().alias("item_total_revenue"),

        pl.col("discount").mean().alias("item_avg_discount"),

        pl.col("updated_date").max().alias("item_last_time"),
    ])
    .with_columns([
        (
            pl.col("item_purchase_count") /
            (pl.col("item_view_count") + 1)
        ).alias("item_purchase_per_view"),

        (
            pl.col("item_cart_count") /
            (pl.col("item_view_count") + 1)
        ).alias("item_cart_per_view"),

        (
            pl.col("item_purchase_count") /
            (pl.col("item_cart_count") + 1)
        ).alias("item_purchase_per_cart"),

        (
            (pl.lit(rec_time_train) - pl.col("item_last_time"))
            .dt.total_seconds() / 86400
        ).alias("item_recency_days"),
    ])
    .drop("item_last_time")
)

item_features.sink_parquet(
    "/kaggle/working/item_features_train.parquet",
    compression="zstd"
)

In [ ]:
print(item_features.head().collect(engine="streaming"))

### User - Items

In [ ]:
# user_item_features = (
#     history_train
#     .group_by(["customer_id", "item_id"])
#     .agg([
#         pl.len().alias("ui_total_events"),

#         (pl.col("event_type") == "view_item").sum().alias("ui_view_count"),
#         (pl.col("event_type") == "add_to_cart").sum().alias("ui_cart_count"),
#         (pl.col("event_type") == "purchase").sum().alias("ui_purchase_count"),

#         pl.col("quantity").sum().alias("ui_total_quantity"),
#         pl.col("event_weight").sum().alias("ui_total_weight"),

#         pl.col("price").mean().alias("ui_avg_price"),
#         pl.col("discount").mean().alias("ui_avg_discount"),

#         pl.col("updated_date").max().alias("ui_last_time"),
#     ])
#     .with_columns([
#         (
#             (pl.lit(rec_time_train) - pl.col("ui_last_time"))
#             .dt.total_seconds() / 86400
#         ).alias("ui_recency_days"),

#         (pl.col("ui_view_count") > 0).cast(pl.Int8).alias("ui_has_viewed"),
#         (pl.col("ui_cart_count") > 0).cast(pl.Int8).alias("ui_has_carted"),
#         (pl.col("ui_purchase_count") > 0).cast(pl.Int8).alias("ui_has_purchased"),
#     ])
#     .drop("ui_last_time")
# )

user_item_features_train = (
    history_train
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_purchase_count"),
        pl.col("quantity").sum().alias("ui_total_quantity"),
        (pl.col("price") * pl.col("quantity")).sum().alias("ui_total_spending"),
        pl.col("price").mean().alias("ui_avg_price"),
        pl.col("discount").mean().alias("ui_avg_discount"),
        pl.col("updated_date").max().alias("ui_last_purchase_time"),
    ])
    .with_columns([
        (
            (pl.lit(rec_time_train) - pl.col("ui_last_purchase_time"))
            .dt.total_seconds() / 86400
        ).alias("ui_recency_days")
    ])
    .drop("ui_last_purchase_time")
)

user_item_features.sink_parquet(
    "/kaggle/working/user_item_features_train.parquet",
    compression="zstd"
)

### Popular items

In [ ]:
# popular_items = (
#     history_train
#     .group_by("item_id")
#     .agg([
#         pl.col("event_weight").sum().alias("pop_score"),
#         (pl.col("event_type") == "purchase").sum().alias("pop_purchase_count"),
#         (pl.col("event_type") == "add_to_cart").sum().alias("pop_cart_count"),
#         (pl.col("event_type") == "view_item").sum().alias("pop_view_count"),
#     ])
#     .sort("pop_score", descending=True)
# )

popular_items = (
    history_train
    .group_by("item_id")
    .agg([
        pl.len().alias("pop_purchase_count"),
        pl.col("quantity").sum().alias("pop_total_quantity"),
        (pl.col("price") * pl.col("quantity")).sum().alias("pop_revenue"),
        pl.col("updated_date").max().alias("item_last_purchase_time"),
    ])
    .sort("pop_purchase_count", descending=True)
)

popular_items.sink_parquet(
    "/kaggle/working/popular_items_train.parquet",
    compression="zstd"
)

### Valid positive

In [ ]:
dec_positive = (
    label_dec
    .filter(pl.col("event_type") == "purchase")
    .select(["customer_id", "item_id"])
    .unique()
    .with_columns(
        pl.lit(1).cast(pl.Int8).alias("label")
    )
)

### LightBGM Candidate

In [ ]:
# N_USERS = 10_000
# TOP_POP = 50

# target_users = (
#     valid_positive
#     .select("customer_id")
#     .unique()
#     # .head(N_USERS)
# )

target_users_train = (
    dec_positive
    .select("customer_id")
    .unique()
)

In [ ]:
TOP_POP = 50
HIST_K = 50

popular_candidates = (
    target_users_train
    .join(
        popular_items.select("item_id").head(TOP_POP),
        how="cross"
    )
    .with_columns([
        pl.lit(1).cast(pl.Int8).alias("cand_from_popular"),
        pl.lit(0).cast(pl.Int8).alias("cand_from_history"),
    ])
)

history_candidates = (
    user_item_features
    .join(target_users_train, on="customer_id", how="inner")
    .with_columns(
        pl.col("ui_total_weight")
        .rank("ordinal", descending=True)
        .over("customer_id")
        .alias("hist_rank")
    )
    .filter(pl.col("hist_rank") <= HIST_K)
    .select(["customer_id", "item_id"])
    .with_columns([
        pl.lit(0).cast(pl.Int8).alias("cand_from_popular"),
        pl.lit(1).cast(pl.Int8).alias("cand_from_history"),
    ])
)

candidates = (
    pl.concat(
        [popular_candidates, history_candidates],
        how="vertical_relaxed"
    )
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.col("cand_from_popular").max(),
        pl.col("cand_from_history").max(),
    ])
)

### Candidate label

In [ ]:
train_pairs = (
    candidates
    .join(
        dec_positive.join(target_users_train, on="customer_id", how="inner"),
        on=["customer_id", "item_id"],
        how="left"
    )
    .with_columns(
        pl.col("label").fill_null(0).cast(pl.Int8)
    )
)

### Train pairs

In [ ]:
train_features = (
    train_pairs
    .join(user_features, on="customer_id", how="left")
    .join(item_features, on="item_id", how="left")
    .join(user_item_features, on=["customer_id", "item_id"], how="left")
    .fill_null(0)
    .with_columns([
        (
            pl.col("item_avg_price") - pl.col("user_avg_price")
        ).alias("price_diff_user_item"),

        (
            pl.col("item_avg_price") / (pl.col("user_avg_price") + 1)
        ).alias("price_ratio_user_item"),

        (
            pl.col("item_avg_discount") - pl.col("user_avg_discount")
        ).alias("discount_diff_user_item"),
    ])
)

train_features.sink_parquet(
    "/kaggle/working/lgbm_train_features_dec.parquet",
    compression="zstd"
)

# Feature Thang 1

In [ ]:
rec_time_jan = datetime(2026, 1, 1)

user_features_jan = (
    history_jan
    .group_by("customer_id")
    .agg([
        pl.len().alias("user_total_events"),
        (pl.col("event_type") == "view_item").sum().alias("user_view_count"),
        (pl.col("event_type") == "add_to_cart").sum().alias("user_cart_count"),
        (pl.col("event_type") == "purchase").sum().alias("user_purchase_count"),
        pl.col("item_id").n_unique().alias("user_unique_items"),
        pl.col("price").mean().alias("user_avg_price"),
        pl.col("price").max().alias("user_max_price"),
        pl.col("price").min().alias("user_min_price"),
        pl.col("quantity").sum().alias("user_total_quantity"),
        pl.col("event_weight").sum().alias("user_total_weight"),
        (pl.col("price") * pl.col("quantity")).sum().alias("user_total_spending"),
        pl.col("discount").mean().alias("user_avg_discount"),
        pl.col("updated_date").max().alias("user_last_time"),
    ])
    .with_columns([
        ((pl.lit(rec_time_jan) - pl.col("user_last_time")).dt.total_seconds() / 86400)
        .alias("user_recency_days"),

        (pl.col("user_purchase_count") / (pl.col("user_total_events") + 1))
        .alias("user_purchase_ratio"),

        (pl.col("user_cart_count") / (pl.col("user_view_count") + 1))
        .alias("user_cart_per_view"),
    ])
    .drop("user_last_time")
)

In [ ]:
item_features_jan = (
    history_jan
    .group_by("item_id")
    .agg([
        pl.len().alias("item_total_events"),
        (pl.col("event_type") == "view_item").sum().alias("item_view_count"),
        (pl.col("event_type") == "add_to_cart").sum().alias("item_cart_count"),
        (pl.col("event_type") == "purchase").sum().alias("item_purchase_count"),
        pl.col("customer_id").n_unique().alias("item_unique_users"),
        pl.col("price").mean().alias("item_avg_price"),
        pl.col("price").max().alias("item_max_price"),
        pl.col("price").min().alias("item_min_price"),
        pl.col("quantity").sum().alias("item_total_quantity"),
        pl.col("event_weight").sum().alias("item_total_weight"),
        (pl.col("price") * pl.col("quantity")).sum().alias("item_total_revenue"),
        pl.col("discount").mean().alias("item_avg_discount"),
        pl.col("updated_date").max().alias("item_last_time"),
    ])
    .with_columns([
        (pl.col("item_purchase_count") / (pl.col("item_view_count") + 1))
        .alias("item_purchase_per_view"),

        (pl.col("item_cart_count") / (pl.col("item_view_count") + 1))
        .alias("item_cart_per_view"),

        (pl.col("item_purchase_count") / (pl.col("item_cart_count") + 1))
        .alias("item_purchase_per_cart"),

        ((pl.lit(rec_time_jan) - pl.col("item_last_time")).dt.total_seconds() / 86400)
        .alias("item_recency_days"),
    ])
    .drop("item_last_time")
)

In [ ]:
# user_item_features_jan = (
#     history_jan
#     .group_by(["customer_id", "item_id"])
#     .agg([
#         pl.len().alias("ui_total_events"),
#         (pl.col("event_type") == "view_item").sum().alias("ui_view_count"),
#         (pl.col("event_type") == "add_to_cart").sum().alias("ui_cart_count"),
#         (pl.col("event_type") == "purchase").sum().alias("ui_purchase_count"),
#         pl.col("quantity").sum().alias("ui_total_quantity"),
#         pl.col("event_weight").sum().alias("ui_total_weight"),
#         pl.col("price").mean().alias("ui_avg_price"),
#         pl.col("discount").mean().alias("ui_avg_discount"),
#         pl.col("updated_date").max().alias("ui_last_time"),
#     ])
#     .with_columns([
#         ((pl.lit(rec_time_jan) - pl.col("ui_last_time")).dt.total_seconds() / 86400)
#         .alias("ui_recency_days"),

#         (pl.col("ui_view_count") > 0).cast(pl.Int8).alias("ui_has_viewed"),
#         (pl.col("ui_cart_count") > 0).cast(pl.Int8).alias("ui_has_carted"),
#         (pl.col("ui_purchase_count") > 0).cast(pl.Int8).alias("ui_has_purchased"),
#     ])
#     .drop("ui_last_time")
# )

user_item_features_jan = (
    history_jan
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_purchase_count"),
        pl.col("quantity").sum().alias("ui_total_quantity"),
        (pl.col("price") * pl.col("quantity")).sum().alias("ui_total_spending"),
        pl.col("price").mean().alias("ui_avg_price"),
        pl.col("discount").mean().alias("ui_avg_discount"),
        pl.col("updated_date").max().alias("ui_last_purchase_time"),
    ])
    .with_columns([
        (
            (pl.lit(rec_time_jan) - pl.col("ui_last_purchase_time"))
            .dt.total_seconds() / 86400
        ).alias("ui_recency_days")
    ])
    .drop("ui_last_purchase_time")
)

In [ ]:
# popular_items_jan = (
#     history_jan
#     .group_by("item_id")
#     .agg([
#         pl.col("event_weight").sum().alias("pop_score"),
#         (pl.col("event_type") == "purchase").sum().alias("pop_purchase_count"),
#         (pl.col("event_type") == "add_to_cart").sum().alias("pop_cart_count"),
#         (pl.col("event_type") == "view_item").sum().alias("pop_view_count"),
#     ])
#     .sort("pop_score", descending=True)
# )

# popular_items_jan = (
#     history_jan
#     .group_by("item_id")
#     .agg([
#         pl.len().alias("pop_purchase_count"),
#         pl.col("quantity").sum().alias("pop_total_quantity"),
#         (pl.col("price") * pl.col("quantity")).sum().alias("pop_revenue"),
#         pl.col("updated_date").max().alias("item_last_purchase_time"),
#     ])
#     .sort("pop_purchase_count", descending=True)
# )

popular_items_jan = (
    history_jan
    .group_by("item_id")
    .agg([
        pl.len().alias("item_purchase_count"),
        pl.col("quantity").sum().alias("item_total_quantity"),
        (pl.col("price") * pl.col("quantity")).sum().alias("item_total_revenue"),
        pl.col("customer_id").n_unique().alias("item_unique_buyers"),
        pl.col("updated_date").max().alias("item_last_purchase_time"),
    ])
    .with_columns([
        (
            (pl.lit(rec_time_jan) - pl.col("item_last_purchase_time"))
            .dt.total_seconds() / 86400
        ).alias("item_recency_days")
    ])
    .with_columns([
        (
            pl.col("item_purchase_count") * 1.0
            + pl.col("item_total_quantity") * 0.2
            + pl.col("item_unique_buyers") * 0.5
        ).alias("pop_score")
    ])
    .sort("pop_score", descending=True)
    .drop("item_last_purchase_time")
)

## Model LightGBM

In [ ]:
feature_cols = [
    # candidate source
    "cand_from_popular",
    "cand_from_history",

    # user features
    "user_total_events",
    "user_view_count",
    "user_cart_count",
    "user_purchase_count",
    "user_unique_items",
    "user_avg_price",
    "user_max_price",
    "user_min_price",
    "user_total_quantity",
    "user_total_weight",
    "user_total_spending",
    "user_avg_discount",
    "user_recency_days",
    "user_purchase_ratio",
    "user_cart_per_view",

    # item features
    "item_total_events",
    "item_view_count",
    "item_cart_count",
    "item_purchase_count",
    "item_unique_users",
    "item_avg_price",
    "item_max_price",
    "item_min_price",
    "item_total_quantity",
    "item_total_weight",
    "item_total_revenue",
    "item_avg_discount",
    "item_purchase_per_view",
    "item_cart_per_view",
    "item_purchase_per_cart",
    "item_recency_days",

    # user-item features
    "ui_total_events",
    "ui_view_count",
    "ui_cart_count",
    "ui_purchase_count",
    "ui_total_quantity",
    "ui_total_weight",
    "ui_avg_price",
    "ui_avg_discount",
    "ui_recency_days",
    "ui_has_viewed",
    "ui_has_carted",
    "ui_has_purchased",

    # interaction features
    "price_diff_user_item",
    "price_ratio_user_item",
    "discount_diff_user_item",
]

In [ ]:
import gc
import lightgbm as lgb
import polars as pl

TRAIN_PATH = "/kaggle/working/lgbm_train_features_dec.parquet"
MODEL_PATH = "/kaggle/working/lgbm_ranker_full_chunked.txt"

N_BUCKETS_TRAIN = 20
TOTAL_BOOST_ROUND = 300
ROUNDS_PER_BUCKET = max(1, TOTAL_BOOST_ROUND // N_BUCKETS_TRAIN)

train_lf = pl.scan_parquet(TRAIN_PATH)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [10],

    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 100,

    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    "lambda_l2": 1.0,

    # Giảm RAM
    "max_bin": 63,
    "force_col_wise": True,
    "histogram_pool_size": 512,

    "verbosity": -1,
    "seed": 42,
}

model = None

for b in range(N_BUCKETS_TRAIN):
    print(f"Training bucket {b + 1}/{N_BUCKETS_TRAIN}")

    part_pd = (
        train_lf
        .filter((pl.col("customer_id").hash() % N_BUCKETS_TRAIN) == b)
        .select(["customer_id", "item_id", "label"] + feature_cols)
        .with_columns([
            pl.col("label").cast(pl.Int8),
            *[pl.col(c).cast(pl.Float32) for c in feature_cols],
        ])
        .sort("customer_id")
        .collect(engine="streaming")
        .to_pandas()
    )

    if len(part_pd) == 0:
        print("Skip empty bucket")
        continue

    if part_pd["label"].sum() == 0:
        print("Skip bucket because no positive label")
        del part_pd
        gc.collect()
        continue

    group = (
        part_pd
        .groupby("customer_id", sort=False)
        .size()
        .to_numpy()
    )

    X_part = part_pd[feature_cols]
    y_part = part_pd["label"]

    print("rows:", len(part_pd))
    print("num groups:", len(group))
    print("positive labels:", y_part.sum())

    dtrain = lgb.Dataset(
        X_part,
        label=y_part,
        group=group,
        free_raw_data=True
    )

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=ROUNDS_PER_BUCKET,
        init_model=model,
        keep_training_booster=True
    )

    model.save_model(MODEL_PATH)

    del part_pd, X_part, y_part, group, dtrain
    gc.collect()

model.save_model(MODEL_PATH)

print("Saved model:", MODEL_PATH)

In [ ]:
import gc
import pickle
import polars as pl
import lightgbm as lgb

MODEL_PATH = "/kaggle/working/lgbm_ranker_full_chunked.txt"
SUBMISSION_PATH = "/kaggle/working/submission_jan.pkl"

model = lgb.Booster(model_file=MODEL_PATH)

TOP_POP = 50
HIST_K = 50
N_BUCKETS_PREDICT = 20

target_users_jan = (
    history_jan
    .select("customer_id")
    .unique()
)

popular_top_items = (
    popular_items_jan
    .select("item_id")
    .head(200)
    .collect(engine="streaming")["item_id"]
    .to_list()
)

def fill_top10(items, popular_items, k=10):
    result = []
    seen = set()

    for item in items:
        if item not in seen:
            result.append(item)
            seen.add(item)
        if len(result) == k:
            return result

    for item in popular_items:
        if item not in seen:
            result.append(item)
            seen.add(item)
        if len(result) == k:
            return result

    return result

submission = {}

for b in range(N_BUCKETS_PREDICT):
    print(f"Predict bucket {b + 1}/{N_BUCKETS_PREDICT}")

    target_users_b = (
        target_users_jan
        .with_columns(
            (pl.col("customer_id").hash() % N_BUCKETS_PREDICT).alias("bucket")
        )
        .filter(pl.col("bucket") == b)
        .drop("bucket")
    )

    popular_candidates_b = (
        target_users_b
        .join(
            popular_items_jan.select("item_id").head(TOP_POP),
            how="cross"
        )
        .with_columns([
            pl.lit(1).cast(pl.Int8).alias("cand_from_popular"),
            pl.lit(0).cast(pl.Int8).alias("cand_from_history"),
        ])
    )

    history_candidates_b = (
        user_item_features_jan
        .join(target_users_b, on="customer_id", how="inner")
        .with_columns(
            # pl.col("ui_total_weight")
            pl.col("ui_total_purchase")
            .rank("ordinal", descending=True)
            .over("customer_id")
            .alias("hist_rank")
        )
        .filter(pl.col("hist_rank") <= HIST_K)
        .select(["customer_id", "item_id"])
        .with_columns([
            pl.lit(0).cast(pl.Int8).alias("cand_from_popular"),
            pl.lit(1).cast(pl.Int8).alias("cand_from_history"),
        ])
    )

    candidates_b = (
        pl.concat(
            [popular_candidates_b, history_candidates_b],
            how="vertical_relaxed"
        )
        .group_by(["customer_id", "item_id"])
        .agg([
            pl.col("cand_from_popular").max(),
            pl.col("cand_from_history").max(),
        ])
    )

    predict_features_b = (
        candidates_b
        .join(user_features_jan, on="customer_id", how="left")
        .join(item_features_jan, on="item_id", how="left")
        .join(user_item_features_jan, on=["customer_id", "item_id"], how="left")
        .fill_null(0)
        .with_columns([
            (
                pl.col("item_avg_price") - pl.col("user_avg_price")
            ).alias("price_diff_user_item"),

            (
                pl.col("item_avg_price") / (pl.col("user_avg_price") + 1)
            ).alias("price_ratio_user_item"),

            (
                pl.col("item_avg_discount") - pl.col("user_avg_discount")
            ).alias("discount_diff_user_item"),
        ])
    )

    pred_pd = (
        predict_features_b
        .select(["customer_id", "item_id"] + feature_cols)
        .with_columns([
            *[pl.col(c).cast(pl.Float32) for c in feature_cols],
        ])
        .collect(engine="streaming")
        .to_pandas()
    )

    if len(pred_pd) == 0:
        continue

    pred_pd["score"] = model.predict(pred_pd[feature_cols])

    top10_b = (
        pred_pd
        .sort_values(["customer_id", "score"], ascending=[True, False])
        .drop_duplicates(["customer_id", "item_id"])
        .groupby("customer_id")
        .head(10)
    )

    sub_b = (
        top10_b
        .groupby("customer_id")["item_id"]
        .apply(list)
        .to_dict()
    )

    users_b = (
        target_users_b
        .collect(engine="streaming")["customer_id"]
        .to_list()
    )

    for customer_id in users_b:
        items = sub_b.get(customer_id, [])
        submission[customer_id] = fill_top10(items, popular_top_items, k=10)

    # Lưu tạm sau mỗi bucket để tránh mất kết quả nếu notebook crash
    with open(SUBMISSION_PATH, "wb") as f:
        pickle.dump(submission, f)

    del pred_pd, top10_b, sub_b
    gc.collect()

with open(SUBMISSION_PATH, "wb") as f:
    pickle.dump(submission, f)

print("Saved:", SUBMISSION_PATH)
print("Num users:", len(submission))

sample_user = next(iter(submission))
print(sample_user, submission[sample_user], len(submission[sample_user]))

### Create submission

In [ ]:
# with open("/kaggle/working/submission_jan.pkl", "wb") as f:
#     pickle.dump(submission, f)

# print("Saved:", "/kaggle/working/submission_jan.pkl")
# print("Num users:", len(submission))

# sample_user = next(iter(submission))
# print(sample_user, submission[sample_user], len(submission[sample_user]))

# with open(SUBMISSION_PATH, "wb") as f:
#     pickle.dump(submission, f)

# print("Saved:", SUBMISSION_PATH)
# print("Num users:", len(submission))

# sample_user = next(iter(submission))
# print(sample_user, submission[sample_user], len(submission[sample_user]))